In [1]:
from datasets import load_dataset
import torch
from torch import nn
from datasets import Dataset
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix,ConfusionMatrixDisplay
from transformers import DataCollatorForTokenClassification
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForTokenClassification
import evaluate
from transformers import TrainingArguments, Trainer


c:\anacondaa\envs\tne\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:


dataset = load_dataset("unimelb-nlp/wikiann", "tr")
print(dataset)
print(dataset["train"][:20])
print(dataset["validation"][:20])

DatasetDict({
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 10000
    })
    train: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 20000
    })
})
{'tokens': [['3.lük', 'maçında', 'Slovenya', 'Millî', 'Basketbol', "Takımı'nı", 'yendikleri', 'maçta', '23', 'sayı', ',', '6', 'ribaund', ',', '2', 'blok', 'istatistikleriyle', 'oynamış', 've', '12', 'faul', 'yaptırmıştır', '.'], ["'", "''", 'Denizlispor', "''", "'"], ['Hami', 'Mandıralı', '36', ',', 'Orhan', 'Çıkırıkçı', '46', ',', '48', ',', 'Arçil', 'Arveladze', '70'], ['San', 'Antonio', 'Spurs', '(', "Milwaukee'den", ')'], ['Divandere', '(', 'Dîwandere', ')'], ['Büyük', 'Ermenistan', 'kurma', 'girişimleri', 'sona', 'ermiştir', '.'], ['YÖNLENDİRME', '2010-11', '1.', 'Lig'], ['David', 'Ferrer', '(', '16', ')'], ['Hollywood', 'için', 

In [3]:

model_name = "dbmdz/bert-base-turkish-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)


In [4]:
def tokenize_and_align_labels(examples):
    
    tokenized = tokenizer(
        examples["tokens"], 
        truncation=True, 
        is_split_into_words=True
    )
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        pre_word_idx = None
        label_id = []
        for word_idx in word_ids:
            if word_idx is None:
                
                label_id.append(-100)
            elif word_idx != pre_word_idx:
                
                label_id.append(label[word_idx])
            else:
                
                label_id.append(-100)
            pre_word_idx = word_idx
        
        labels.append(label_id)
    
    tokenized["labels"] = labels
    return tokenized

In [5]:
tokenized_set = dataset.map(
    tokenize_and_align_labels,
    batched=True
)

In [6]:
tokenized_set = tokenized_set.remove_columns(["tokens", "ner_tags", "langs", "spans"])

In [7]:
BATCH_SIZE=32
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
train_set=DataLoader(tokenized_set['train'],batch_size=BATCH_SIZE,shuffle=True,collate_fn=data_collator)
test_set=DataLoader(tokenized_set['test'],batch_size=BATCH_SIZE,shuffle=False,collate_fn=data_collator)
validation_set=DataLoader(tokenized_set['validation'],batch_size=BATCH_SIZE,shuffle=False,collate_fn=data_collator)

In [8]:

label_names = dataset["train"].features["ner_tags"].feature.names

id2label = {i: label for i, label in enumerate(label_names)}
label2id = {label: i for i, label in enumerate(label_names)}




In [9]:
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=7,           
    id2label=id2label,      
    label2id=label2id      
)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 3160.25it/s]
[transformers] BertForTokenClassification LOAD REPORT from: dbmdz/bert-base-turkish-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/arch

In [10]:

metric = evaluate.load("seqeval")

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    
    predictions = np.argmax(logits, axis=-1)

    
    true_labels = [
        [label_names[l] for l in label if l != -100]
        for label in labels
    ]
    
    
    true_predictions = [
        [label_names[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    
    all_metrics = metric.compute(predictions=true_predictions, references=true_labels)
    
    return {
        "precision": all_metrics["overall_precision"],
        "recall": all_metrics["overall_recall"],
        "f1": all_metrics["overall_f1"],
        "accuracy": all_metrics["overall_accuracy"],
    }

In [11]:

args = TrainingArguments(
    output_dir="./berturk-ner-model", 
    eval_strategy="epoch",      
    learning_rate=2e-5,              
    per_device_train_batch_size=16,   
    per_device_eval_batch_size=16,    
    num_train_epochs=3,               
    weight_decay=0.01,               
    save_strategy="epoch"             
)


trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_set["train"],       
    eval_dataset=tokenized_set["validation"],    
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)


trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.156959,0.120027,0.893685,0.909817,0.901679,0.966048
2,0.093808,0.114944,0.910684,0.922542,0.916575,0.970354
3,0.060803,0.122330,0.914540,0.927869,0.921156,0.971303


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.62it/s]


TrainOutput(global_step=3750, training_loss=0.12480155512491863, metrics={'train_runtime': 663.423, 'train_samples_per_second': 90.44, 'train_steps_per_second': 5.653, 'total_flos': 925587613467264.0, 'train_loss': 0.12480155512491863, 'epoch': 3.0})

In [12]:
test_results = trainer.evaluate(tokenized_set["test"])

Training Loss,Validation Loss,Epoch,Precision,Recall,F1,Accuracy
0.060803,0.123274,3,0.914363,0.922573,0.918450,0.971069


In [13]:
from transformers import pipeline


ner_pipeline = pipeline(
    "ner", 
    model=model, 
    tokenizer=tokenizer, 
    aggregation_strategy="simple",
    device=-1 
)

In [14]:
test_sentences = [
    
    "Mustafa Kemal Atatürk Ankara'da yaşamıştır.",
    
    
    "Çankaya Üniversitesi, Ankara Büyükşehir Belediyesi ile ortak bir proje yürütüyor.",
    
    
    "RKSOFT şirketinde derin öğrenme modelleri ve bilgisayarlı görü algoritmaları geliştiriliyor.",
    
    
    "TEKNOFEST yarışması bu yıl Adana'da büyük bir coşkuyla düzenlenecek.",
    
    
    "IEEE öğrenci kolu, mühendislik fakültesinde yeni bir toplantı organize etti.",
    
    
    "Hasan Kalyoncu Üniversitesi'ne kayıt yaptırmak için Gaziantep'e gittim.",
    
    
    "Elon Musk, SpaceX ve Tesla'nın operasyon merkezini Teksas'a taşıma kararı aldı.",
    
    
    "Prof. Dr. İlber Ortaylı, Topkapı Sarayı'nda Osmanlı tarihi üzerine bir konferans verecek.",
    
    
    "Mark Zuckerberg, Meta'nın gelecekteki sanal gerçeklik yatırımları hakkında konuştu.",
    
    
    "Sağlık Bakanlığı ile Milli Eğitim Bakanlığı pandemiden sonra yeni bir genelge yayınladı.",
    
    
    "Birleşmiş Milletler, New York'taki genel merkezinde acil bir güvenlik zirvesi topladı.",
    
    
    "Anadolu Ajansı'nın son dakika haberine göre, Japonya'nın başkenti Tokyo'da deprem oldu.",
    
    
    "Ali, Ayşe ile birlikte akşam Kızılay Meydanı'nda buluşup kahve içecek.",
    
    
    "Fatih Sultan Mehmet, 1453 yılında İstanbul'u fethederek bir çağı kapattı.",
    
    
    "Galatasaray, UEFA Şampiyonlar Ligi grup maçında Bayern Münih ile karşılaşacak.",
    
    
    "Tarkan'ın yeni albümü sadece Türkiye'de değil, Avrupa'da da büyük ilgi gördü.",
    
   
    "Karadeniz Bölgesi'nde yaz aylarında başlayan çay hasadı sonbahara kadar sürer.",
    
    
    "Ufuk, yarın sabah erkenden Türk Hava Yolları uçağıyla İzmir'e uçacak.",
    
    
    "Boğaziçi Üniversitesi'ndeki araştırmacılar, TÜBİTAK destekli projelerini tamamladı.",
    
    
    "Hafta sonu Kadıköy'den vapura binip Beşiktaş'a geçtik ve Dolmabahçe'yi gezdik."
]

In [17]:
for sentence in test_sentences:
    print(f"sentence: {sentence}")
    results=ner_pipeline(sentence)

    if not results:
        print(f"entity is not found.")
    else:
        for result in results:
            print(f"-{result['word']}: {result['entity_group']} (Score: %{result['score']*100:.1f})")
    print(f"-" * 50) 

sentence: Mustafa Kemal Atatürk Ankara'da yaşamıştır.
-Mustafa Kemal Atatürk: PER (Score: %99.6)
-Ankara: LOC (Score: %99.8)
--------------------------------------------------
sentence: Çankaya Üniversitesi, Ankara Büyükşehir Belediyesi ile ortak bir proje yürütüyor.
-Çankaya Üniversitesi: ORG (Score: %99.8)
-Ankara Büyükşehir Belediyesi: ORG (Score: %95.3)
--------------------------------------------------
sentence: RKSOFT şirketinde derin öğrenme modelleri ve bilgisayarlı görü algoritmaları geliştiriliyor.


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


-görü: PER (Score: %47.6)
--------------------------------------------------
sentence: TEKNOFEST yarışması bu yıl Adana'da büyük bir coşkuyla düzenlenecek.
-##OFE: ORG (Score: %66.4)
-Adana: LOC (Score: %99.9)
--------------------------------------------------
sentence: IEEE öğrenci kolu, mühendislik fakültesinde yeni bir toplantı organize etti.
-IEEE: ORG (Score: %98.5)
--------------------------------------------------
sentence: Hasan Kalyoncu Üniversitesi'ne kayıt yaptırmak için Gaziantep'e gittim.
-Hasan Kalyoncu Üniversitesi: ORG (Score: %99.6)
-Gaziantep: LOC (Score: %99.8)
--------------------------------------------------
sentence: Elon Musk, SpaceX ve Tesla'nın operasyon merkezini Teksas'a taşıma kararı aldı.
-Elon Musk: PER (Score: %90.1)
-SpaceX: ORG (Score: %97.5)
-Tesla: ORG (Score: %96.0)
-Teksas: LOC (Score: %99.3)
--------------------------------------------------
sentence: Prof. Dr. İlber Ortaylı, Topkapı Sarayı'nda Osmanlı tarihi üzerine bir konferans verecek.
-İlber 